In [ ]:
from google.colab import drive
drive.mount("<mount-point>")

In [ ]:
from sklearn.model_selection import KFold

import pandas as pd
import numpy as np
import datasets
import torch

import warnings
warnings.simplefilter(action='ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Preparing data

The current order is:
- 0: zero
- 1: sft
- 2: berzak

In [ ]:
rank_df_ord = pd.read_excel('<project-data-path>', index_col = 0)

rank_df_ord = rank_df_ord.rename({
    'article': 'passage',
    'berzak_distractor': 'distractor_1',
    'zero_distractor': 'distractor_2',
    'sft_distractor': 'distractor_3',
    'berzak': 'distractor_1_com',
    'zero': 'distractor_2_com',
    'sft': 'distractor_3_com'
}, axis = 1)

rank_df_ord.head(1)

## Loading modules

❗ REMEMBER TO UNCOMMENT THE MODEL LOADING PART OF BARTSCORE.

### BLEU & ROUGE (F)

In [ ]:
!pip install sacrebleu
import sacrebleu

!pip install rouge_score
from rouge_score import rouge_scorer

bleu = sacrebleu.sentence_bleu(
    'this is a distractor',
    ['okay this is another span']
    )

print(bleu.score)

rouge_scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)
# Rouge_1_f

print(rouge_scorer.score(
    'this is a distractor',
    'this is a long span'
    )['rouge1'].fmeasure)

### BERTScore (F)

In [ ]:
!pip install evaluate
from evaluate import load

!pip install bert_score
!pip install bart_score
bert_score = load("bertscore")

In [ ]:
from bert_score import score

bert_results_berzak_passage = bert_score.compute(
    predictions=rank_df_ord['distractor_1'].to_list(),
    references=rank_df_ord['a_span'].to_list(),
    model_type="microsoft/deberta-large-mnli"
    )
bert_results_sft_passage = bert_score.compute(
    predictions=rank_df_ord['distractor_3'].to_list(),
    references=rank_df_ord['a_span'].to_list(),
    model_type="microsoft/deberta-large-mnli"
    )
bert_results_zero_passage = bert_score.compute(
    predictions=rank_df_ord['distractor_2'].to_list(),
    references=rank_df_ord['a_span'].to_list(),
    model_type="microsoft/deberta-large-mnli"
    )

bert_scores = {
    'berzak': bert_results_berzak_passage,
    'zero': bert_results_zero_passage,
    'sft': bert_results_sft_passage
}

pd.to_pickle(bert_scores, '<project-data-path>')

### BARTScore

In [ ]:
import torch
import torch.nn as nn
import traceback
from transformers import BartTokenizer, BartForConditionalGeneration
from typing import List
import numpy as np


class BARTScorer:
    def __init__(self, device='cuda:0', max_length=1024, checkpoint='facebook/bart-large-cnn'):
        # Set up model
        self.device = device
        self.max_length = max_length
        self.tokenizer = BartTokenizer.from_pretrained(checkpoint)
        self.model = BartForConditionalGeneration.from_pretrained(checkpoint)
        self.model.eval()
        self.model.to(device)

        # Set up loss
        self.loss_fct = nn.NLLLoss(reduction='none', ignore_index=self.model.config.pad_token_id)
        self.lsm = nn.LogSoftmax(dim=1)

    def load(self, path=None):
        """ Load model from paraphrase finetuning """
        if path is None:
            path = 'models/bart.pth'
        self.model.load_state_dict(torch.load(path, map_location=self.device))

    def score(self, srcs, tgts, batch_size=4):
        """ Score a batch of examples """
        score_list = []
        for i in range(0, len(srcs), batch_size):
            src_list = srcs[i: i + batch_size]
            tgt_list = tgts[i: i + batch_size]
            try:
                with torch.no_grad():
                    encoded_src = self.tokenizer(
                        src_list,
                        max_length=self.max_length,
                        truncation=True,
                        padding=True,
                        return_tensors='pt'
                    )
                    encoded_tgt = self.tokenizer(
                        tgt_list,
                        max_length=self.max_length,
                        truncation=True,
                        padding=True,
                        return_tensors='pt'
                    )
                    src_tokens = encoded_src['input_ids'].to(self.device)
                    src_mask = encoded_src['attention_mask'].to(self.device)

                    tgt_tokens = encoded_tgt['input_ids'].to(self.device)
                    tgt_mask = encoded_tgt['attention_mask']
                    tgt_len = tgt_mask.sum(dim=1).to(self.device)

                    output = self.model(
                        input_ids=src_tokens,
                        attention_mask=src_mask,
                        labels=tgt_tokens
                    )
                    logits = output.logits.view(-1, self.model.config.vocab_size)
                    loss = self.loss_fct(self.lsm(logits), tgt_tokens.view(-1))
                    loss = loss.view(tgt_tokens.shape[0], -1)
                    loss = loss.sum(dim=1) / tgt_len
                    curr_score_list = [-x.item() for x in loss]
                    score_list += curr_score_list

            except RuntimeError:
                traceback.print_exc()
                print(f'source: {src_list}')
                print(f'target: {tgt_list}')
                exit(0)
        return score_list

    def multi_ref_score(self, srcs, tgts: List[List[str]], agg="mean", batch_size=4):
        # Assert we have the same number of references
        ref_nums = [len(x) for x in tgts]
        if len(set(ref_nums)) > 1:
            raise Exception("You have different number of references per test sample.")

        ref_num = len(tgts[0])
        score_matrix = []
        for i in range(ref_num):
            curr_tgts = [x[i] for x in tgts]
            scores = self.score(srcs, curr_tgts, batch_size)
            score_matrix.append(scores)
        if agg == "mean":
            score_list = np.mean(score_matrix, axis=0)
        elif agg == "max":
            score_list = np.max(score_matrix, axis=0)
        else:
            raise NotImplementedError
        return list(score_list)

    def test(self, batch_size=3):
        """ Test """
        src_list = [
            'This is a very good idea. Although simple, but very insightful.',
            'Can I take a look?',
            'Do not trust him, he is a liar.'
        ]

        tgt_list = [
            "That's stupid.",
            "What's the problem?",
            'He is trustworthy.'
        ]

        print(self.score(src_list, tgt_list, batch_size))

In [ ]:
bart_scorer = BARTScorer(device=device, checkpoint='facebook/bart-large-cnn')

# !pip install gdown
# !gdown 1_7JfF7KOInb7ZrxKHIigTMR4ChVET01m -O <project-data-path>

bart_scorer.load(path='<project-data-path>')

bart_scorer.score(
    ['this is a distractor'],
    ['this is a long span']
)[0]

In [ ]:
bart_results_berzak_passage = bart_scorer.score(
    rank_df_ord['distractor_1'].to_list(),
    rank_df_ord['a_span'].to_list(),
    # model_type="microsoft/deberta-large-mnli"
    )
bart_results_sft_passage = bart_scorer.score(
    rank_df_ord['distractor_3'].to_list(),
    rank_df_ord['a_span'].to_list(),
    # model_type="microsoft/deberta-large-mnli"
    )
bart_results_zero_passage = bart_scorer.score(
    rank_df_ord['distractor_2'].to_list(),
    rank_df_ord['a_span'].to_list(),
    # model_type="microsoft/deberta-large-mnli"
    )

bart_scores = {
    'berzak': bart_results_berzak_passage,
    'zero': bart_results_zero_passage,
    'sft': bart_results_sft_passage
}

pd.to_pickle(bart_scores, '<project-data-path>')

### BLEURT

In [ ]:
!pip install git+https://github.com/google-research/bleurt.git

# !pip install evaluate
from evaluate import load
bleurt = load('bleurt', 'bleurt-large-512')

In [ ]:
bleurt.compute(
    predictions=['this is a long distractor'],
    references=['this is a short span']
)['scores']

In [ ]:
bleurt_results_berzak_passage = bleurt.compute(
    predictions=rank_df_ord['distractor_1'].to_list(),
    references=rank_df_ord['a_span'].to_list(),
    # model_type="microsoft/deberta-large-mnli"
    )
bleurt_results_sft_passage = bleurt.compute(
    predictions=rank_df_ord['distractor_3'].to_list(),
    references=rank_df_ord['a_span'].to_list(),
    # model_type="microsoft/deberta-large-mnli"
    )
bleurt_results_zero_passage = bleurt.compute(
    predictions=rank_df_ord['distractor_2'].to_list(),
    references=rank_df_ord['a_span'].to_list(),
    # model_type="microsoft/deberta-large-mnli"
    )

bleurt_scores = {
    'berzak': bleurt_results_berzak_passage,
    'zero': bleurt_results_zero_passage,
    'sft': bleurt_results_sft_passage
}

pd.to_pickle(bleurt_scores, '<project-data-path>')

### NLI

In [ ]:
# !pip install -U "huggingface_hub[cli]"
# !hf download microsoft/deberta-large-mnli --local-dir <project-data-path> --cache-dir /tmp/cache

In [ ]:
device

In [ ]:
from transformers import pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipe = pipeline(
    model='microsoft/deberta-large-mnli',
    return_all_scores=True,
    device=device
    )

pipe({
    'text': 'this is a span, quite long one',
    'text_pair': 'okay, this is the distractor based on the given span'
})

## Compiling all metrics

In [ ]:
a_spans = rank_df_ord['a_span_full'].values
passages = rank_df_ord['passage'].values

berzak_b_com, zero_b_com, sft_com = zip(*rank_df_ord[['distractor_1_com', 'distractor_2_com', 'distractor_3_com']].values)
berzak_b, zero_b, sft_b = zip(*rank_df_ord[['distractor_1', 'distractor_2', 'distractor_3']].values)

demo_span = a_spans[1]
demo_b = berzak_b[1]

print(demo_span)
print(demo_b)

In [ ]:
def get_textual_overlap(distractor, span):
    bleu_score = sacrebleu.sentence_bleu(
        distractor, [span]
    ).score

    rouge_score = rouge_scorer.score(
        distractor, span
    )['rouge1'].fmeasure

    return bleu_score, rouge_score

get_textual_overlap(demo_b, demo_span)

In [ ]:
# def get_semantic_similarity(distractor, span):
#     # P, R, F1 = score(
#     #     cands=[distractor],
#     #     refs=[span],
#     #     # model_type="microsoft/deberta-large-mnli",
#     #     lang="en",
#     #     verbose=True
#     # )
#     # bert_score_f = F1.mean().item()
#     bert_score_f = NULL

#     bart_score = bart_scorer.score(
#         [distractor],
#         [span]
#     )[0]

#     bleurt_score = bleurt.compute(
#         predictions=[distractor],
#         references=[span]
#     )['scores'][0]

#     return bert_score_f, bart_score, bleurt_score

# get_semantic_similarity(demo_b, demo_span)

In [ ]:
from transformers import pipeline

def get_logical_inference(distractor, span):
    res = pipe({
        'text': distractor,
        'text_pair': span
    })
    contra_score, neutral_score, entail_score = [i['score'] for i in res]
    # print(res)
    return contra_score, neutral_score, entail_score

get_logical_inference(demo_b, demo_span)

In [ ]:
from tqdm import tqdm
from collections import defaultdict
import json

def get_metric_dict(distractor, span, com_distractor, passage):
    bleu_score, rouge_score = get_textual_overlap(distractor, span)
    # bert_score_f, bart_score, bleurt_score = get_semantic_similarity(distractor, span)
    contra_score, neutral_score, entail_score = get_logical_inference(distractor, span)

    metric_dict = {
        'bleu': bleu_score, 'rouge': rouge_score,
        # 'bert_s': bert_score_f, 'bart_s': bart_score, 'bleurt': bleurt_score,
        'entail': entail_score, 'neutral': neutral_score, 'contra': contra_score
    }

    return metric_dict

# get_metric_dict(demo_b, demo_span)

In [ ]:
rank_df_ord.head(1)

In [ ]:
# all_metrics = defaultdict(dict)

# for i, row in tqdm(rank_df_ord.iterrows()):
#     idx_key = row['index']
#     span = row['a_span']
#     berzak_b, zero_b, sft_b = row[['distractor_1', 'distractor_2', 'distractor_3']]

#     zero_metric_dict = get_metric_dict(zero_b, span)
#     sft_metric_dict = get_metric_dict(sft_b, span)
#     berzak_metric_dict = get_metric_dict(berzak_b, span)

#     all_metrics[idx_key] = {
#         'zero': {
#             'pair': (span, zero_b),
#             'metrics': zero_metric_dict
#         },
#         'sft': {
#             'pair': (span, sft_b),
#             'metrics': sft_metric_dict
#         },
#         'berzak': {
#             'pair': (span, berzak_b),
#             'metrics': berzak_metric_dict
#         }
#     }

# all_metrics

In [ ]:
import json
import os
from collections import defaultdict
from tqdm.auto import tqdm
import torch
from concurrent.futures import ThreadPoolExecutor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. 配置参数
BATCH_SIZE = 8  # 可调整批大小
RESUME = True    # 是否从断点继续
OUTPUT_FILE = "<project-data-path>"
CHECKPOINT_FILE = "<project-data-path>"

# 2. 初始化数据结构
if RESUME and os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        all_metrics = defaultdict(dict, json.load(f))
    processed_indices = set(all_metrics.keys())
else:
    all_metrics = defaultdict(dict)
    processed_indices = set()

# 3. GPU加速的批量处理函数
def process_batch(batch_rows, device='cuda'):
    batch_results = {}

    with ThreadPoolExecutor() as executor:
        futures = []
        for _, row in batch_rows:
            if str(row['index']) in processed_indices:
                continue

            futures.append(executor.submit(
                process_single_row,
                row,
                device=device
            ))

        for future in tqdm(futures, desc="Processing batch"):
            idx_key, result = future.result()
            batch_results[idx_key] = result

    return batch_results

def process_single_row(row, device='cuda'):
    idx_key = row['index']
    passage = row['passage']
    span = row['a_span']
    # full_span = row['a_span_full']
    distractors = row[['distractor_1', 'distractor_2', 'distractor_3']]
    com_distractors = row[['distractor_1_com', 'distractor_2_com', 'distractor_3_com']]

    # 使用GPU加速的metric计算（假设get_metric_dict支持GPU）
    with torch.cuda.device(device):
        metrics = {
            'berzak': {'pair': (span, distractors[0]), 'metrics': get_metric_dict(
                distractors[0], span,
                com_distractors[0], passage
                )},
            'zero': {'pair': (span, distractors[1]), 'metrics': get_metric_dict(
                distractors[1], span,
                com_distractors[1], passage
                )},
            'sft': {'pair': (span, distractors[2]), 'metrics': get_metric_dict(
                distractors[2], span,
                com_distractors[2], passage
                )}
        }

    return idx_key, metrics

# 4. 分批处理主循环
try:
    batch = []
    for i, row in tqdm(rank_df_ord.iterrows(), total=len(rank_df_ord)):
        if str(row['index']) in processed_indices:
            continue

        batch.append((i, row))

        if len(batch) >= BATCH_SIZE:
            batch_results = process_batch(batch)
            all_metrics.update(batch_results)

            # 实时保存结果
            with open(OUTPUT_FILE, 'w') as f:
                json.dump(all_metrics, f)

            # 保存检查点
            with open(CHECKPOINT_FILE, 'w') as f:
                json.dump({'last_index': i}, f)

            batch = []

    # 处理剩余数据
    if batch:
        batch_results = process_batch(batch)
        all_metrics.update(batch_results)
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(all_metrics, f)

except Exception as e:
    print(f"Error occurred: {str(e)}")
    print("Saving progress before exiting...")
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(all_metrics, f)
    raise

# 5. 最终输出
print(f"Processing completed. Results saved to {OUTPUT_FILE}")